# ചെലവ് അവകാശ വാദം വിശ്ലേഷണം

ഈ നോട്ട്‌ബുക്ക് ലോക്കൽ റിസീറ്റ് ചിത്രങ്ങളിൽ നിന്നുള്ള യാത്രാ ചെലവുകൾ പ്രോസസ്സ് ചെയ്യുന്നതിനായി പ്ലഗിനുകൾ ഉപയോഗിക്കുന്ന ഏജന്റ്സിനെ എങ്ങനെ സൃഷ്ടിക്കാമെന്ന്, ചെലവ് അവകാശ ഇമെയിൽ നിർമ്മിക്കാമെന്ന്, പൈ ചാർട്ട് ഉപയോഗിച്ച് ചെലവ് ഡാറ്റയെ വീക്ഷിക്കാമെന്നും കാണിക്കുന്നു. ഏജന്റുകൾ പ്രവർത്തി സാഹചര്യത്തിന്റെ അടിസ്ഥാനത്തിൽ ഫങ്ഷനുകൾ ഡൈനാമിക്കായി തിരഞ്ഞെടുക്കുന്നു.

ഘട്ടങ്ങൾ:
1. OCR ഏജന്റ് ലോക്കൽ റിസീറ്റ് ചിത്രം പ്രോസസ് ചെയ്ത് യാത്രാ ചെലവ് ഡാറ്റ എടുക്കുന്നു.
2. ഇമെയിൽ ഏജന്റ് ചെലവ് അവകാശ ഇമെയിൽ സൃഷ്ടിക്കുന്നു.

### ഒരു യാത്രാ ചെലവ് സാഹചര്യത്തിന്റെ ഉദാഹരണം:
നിങ്ങൾ മറ്റൊരു നഗരത്തിൽ ബിസിനസ് മീറ്റിങ്ങിനായി യാത്ര ചെയ്യുന്ന ഒരു ജീവനക്കാരനാണെന്ന് تصورിക്കുക. നിങ്ങളുടെ കമ്പനി എല്ലാ അനുകൂല യാത്രാ സംബന്ധമായ ചെലവുകളും പ്രതിഫലിപ്പിക്കുന്ന നയമുണ്ട്. യാത്രാ ചെലവുകളുടെ വിഭജനമാണ് ഇതാ:
- ഗതാഗതം:
നിങ്ങളുടെ വീട്ടുനഗരത്തിൽ നിന്ന് ലക്ഷ്യ നഗരത്തിലേക്ക് റൗണ്ട് ട്രിപ്പ് എയർഫെയർ.
എയർപോർട്ടിലേക്കും തിരിച്ചും ടാക്സി അല്ലെങ്കിൽ റൈഡ്-ഹെയ്‌ലിംഗ് സേവനങ്ങൾ.
ലക്ഷ്യ നഗരത്തിൽ പ്രാദേശിക ഗതാഗതം (പബ്ലിക്ക് ട്രാൻസിറ്റ്, കാറുൾപ്പടി, ടാക്സികൾ പോലുള്ളവ).

- താമസം:
മീറ്റിങ് വേദിക്ക് അടുത്തുള്ള മിഡ്-റേഞ്ച് ബിസിനസ് ഹോട്ടലിൽ മൂന്നു രാത്രികൾ താമസിക്കുക.

- ഭക്ഷണം:
കമ്പനിയുടെയും പാടിയുമുള്ളദിവസവേതന നയത്തിന്റെ അടിസ്ഥാനത്തിൽ രാവില, ഉച്ച, രാത്രി ഭക്ഷണത്തിന് പ്രതിദിന അലവൻസ്.

- വിവിധ ചെലവുകൾ:
എയർപോർട്ടിലെ പാർക്കിംഗ് ഫീസുകൾ.
ഹോട്ടലിലെ ഇന്റർനെറ്റ് ആക്‌സസ് ചാർജുകൾ.
ടിപ്പുകൾ അല്ലെങ്കിൽ ചെറിയ സേവന ചാർജുകൾ.

- രേഖകൾ:
നിങ്ങൾ എല്ലാ റിസീറ്റുകളും (ഫ്ലൈറ്റുകൾ, ടാക്സികൾ, ഹോട്ടൽ, ഭക്ഷണം മുതലായവ) ശേഖരിച്ച് പൂർണ്ണമായ ചെലവ് റിപ്പോർട്ട് സമർപ്പിക്കുന്നു.


## ആവശ്യമായ ലൈബ്രററികൾ ഇമ്പോർട്ട് ചെയ്യുക

നോട്ട്ബുക്കിന് ആവശ്യമായ ലൈബ്രററികളും മോഡിയൂളുകളും ഇമ്പോർട്ട് ചെയ്യുക.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## ചെലവ് മോഡലുകൾ നിർവചിക്കുക

 വ്യക്തിഗത ചെലവുകൾക്കുള്ള പിഡാന്റിക് മോഡൽ ഒരു രൂപത്തിൽ സൃഷ്ടിച്ച്, ഉപയോക്തൃ ചോദ്യം ഘടിത ചെലവു ഡാറ്റയിലേക്ക് മാറ്റുന്നതിനുള്ള ExpenseFormatter ക്ലാസ് സൃഷ്ടിക്കുക.

 ഓരോ ചെലവും താഴെയുള്ള ഫോർമാറ്റിൽ പ്രതിനിധീകരിക്കും:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## ഉപകരണങ്ങൾ നിർവചിക്കുക - ഇമെയിൽ സൃഷ്ടിക്കുന്നു

ചെലവു് ക്ലെയിം സമർപ്പിക്കുന്നതിന് ഐമെയിൽ സൃഷ്ടിക്കാൻ ടൂൾ ഫംഗ്ഷൻ ഉണ്ടാക്കുക.
- ഈ ടൂൾ മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് മുതൽ `@tool` ഡെക്കൊറേറ്റർ ഉപയോഗിക്കുന്നു.
- ഇത് ചെലവുകളുടെ മൊത്തം തുക കണക്കാക്കി വിശദാംശങ്ങൾ ഇമെയിൽ ബോഡിയിൽ ഫോർമാറ്റ് ചെയ്യുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# റസീറ്റ് ചിത്രങ്ങളിൽ നിന്നും യാത്ര ചിലവുകൾ മുകളിൽ എടുക്കുന്നതിനുള്ള ടൂൾ

റസീറ്റ് ചിത്രങ്ങളിൽ നിന്നുള്ള യാത്ര ചിലവുകൾ മുകളിൽ എടുക്കാൻ ഒരു ടൂൾ ഫങ്ഷൻ സൃഷ്ടിക്കുക.
- ഈ ടൂൾ മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് നിന്നുള്ള `@tool` ഡെക്കറേറ്റർ ഉപയോഗിക്കുന്നു.
- ഇത് റസീറ്റ് ചിത്രം വായിച്ച്, base64 ആയി എൺകോഡ് ചെയ്ത്, ഏജന്റ് വിശകലനം ചെയ്യുന്നതിനായി ഡാറ്റാ URI നൽകുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ചെലവുകൾ പ്രോസസ്സ് ചെയ്യൽ

ഏജന്റുകൾ നിർവചിച്ച് അവയെ `WorkflowBuilder` ഉപയോഗിച്ച് ഒരു ക്രമചെയ്യപ്പെട്ട പ്രവൃത്തി പ്രക്രിയയിലേക്കു ബന്ധിപ്പിക്കുക.
- ഓസിആർ (OCR) ഏജന്റ് വിൽപ്പനക്കരട് ചിത്രം മുതലെടുക്കുന്ന `load_receipt_image` ഉപകരണം ഉപയോഗിച്ച് ഘടിത ചെലവുകളുടെ ഡാറ്റ പുറത്തെടുക്കുന്നു.
- ഇമെയിൽ ഏജന്റ് പുറപ്പെടുവിച്ച ഡാറ്റ സ്വീകരിച്ച് `generate_expense_email` ഉപകരണം ഉപയോഗിച്ച് ഒരു പ്രൊഫഷണൽ ചെലവ് പരാതി ഇമെയിൽ സൃഷ്ടിക്കുന്നു.
- `WorkflowBuilder` ന്റെ `add_edge` ഉപയോഗിച്ച് ഒരു ക്രമചെയ്യപ്പെട്ട പൈപ്പ്ലൈൻ സൃഷ്ടിക്കുന്നു: OCR ഏജന്റ് → ഇമെയിൽ ഏജന്റ്.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## മുഖ്യ ഫോംഗ്ഷൻ

റസീറ്റ് ഇമേജ് പ്രോസസ്സ് ചെയ്യാനും ചെലവ് കാര്യം ഇമെയിൽ നിർമ്മിക്കാനും സീക്വൻഷ്യൽ വർക്ക്ഫ്ലോ നിർമ്മിച്ച് പ്രവർത്തിപ്പിക്കുക.


> **ഗమనിക്കുക:** ഈ പ്രവൃത്തി പ്രവാഹം നിലവിൽ ആവർത്തനചിത്രം base64 വാചകമായി നൽകുന്നു, gpt-5-mini ഉൾപ്പെടെയുള്ള ഭൂരിഭാഗം ചാറ്റ് മോഡലുകളും ഇതിനെ ചിത്രം ആയി കാണില്ല.
> ഇത് മോഡൽ കോൺടെക്സ്റ്റ് വിംഡോയും കടന്നേക്കാം. Azure AI Vision (അല്ലെങ്കിൽ മറ്റൊരു OCR ടൂൾ) ഉപയോഗിച്ച് OCR പ്രവർത്തിപ്പിക്കുന്നത് എന്നിവ പരാമർശിക്കൂ, മാത്രമല്ല പുറത്തെടുത്ത വാചകം മാത്രം നൽകൂ, അല്ലെങ്കിൽ ചിത്രം `image_url` സന്ദേശമായി അയക്കാൻ പുനർനിർമ്മണം ചെയ്യൂ.
> നിങ്ങൾക്ക് കോൺടെക്സ്റ്റ് പിഴവുകൾ ഒഴിവാക്കണമെങ്കിൽ, ചെറിയ ആവർത്തനചിത്രം ഉപയോഗിക്കുക അല്ലെങ്കിൽ കൂടുതൽ വലിയ കോൺടെക്സ്റ്റ് വിംഡോ ഉള്ള മോഡൽ പരീക്ഷിക്കുക.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
